In [16]:
Ry_ev = 13.6057039763 #eV
value = 0.001* Ry_ev 
print('Value in eV from Ry:', value, 'eV')


Value in eV from Ry: 0.0136057039763 eV


In [13]:
eV_Ry  = 1.0 / Ry_ev
value = 0.05 * eV_Ry
print('Value in Ry from eV:', value, 'Ry')

Value in Ry from eV: 0.0036749292860623626 Ry


In [1]:
from ase.io import read
from ase.visualize import view

view(read('Li2S.relaxed.extxyz'))

<Popen: returncode: None args: ['/home/ameer_ubuntu/miniforge3/envs/qe/bin/p...>

In [4]:
import os
from pathlib import Path

from ase import io
from ase.build import add_vacuum
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.optimize import QuasiNewton

structure_path = Path("input_ads/Li2S_final.extxyz")
pseudo_dir = Path("../PP_NC")
vacuum = 7.5
ecutwfc = 45.0
ecutrho = 4 * ecutwfc
kpts = (1, 1, 1)
fmax = 0.013605704
pseudos = {"Li": "Li_ONCV_PBE-1.2.upf", "S": "S_ONCV_PBE-1.2.upf"
           }
run_dir = Path("./Test") / 'Li2S' / '_RPBE'
run_dir.mkdir(parents=True, exist_ok=True)
atoms = io.read(structure_path)
add_vacuum(atoms, vacuum)
#atoms.rattle(0.005)

os.environ.setdefault("OMP_NUM_THREADS", "1")
profile = EspressoProfile(command="mpirun -n 16 /home/ameer_ubuntu/miniforge3/envs/qe/bin/pw.x", pseudo_dir=str(pseudo_dir))
calc = Espresso(
    profile=profile,
    pseudopotentials=pseudos,
    input_data={
        "control": {"calculation": "scf", "prefix": "adsorbate"},
        "system": {"ecutwfc": ecutwfc, "ecutrho": ecutrho},
        #"vdw_corr": 'dft-d3',
        "input_dft":"rpbe+w32c",
        "occupations": "smearing",
        "smearing": "gaussian",
        "conv_thr": 1.0e-6,
        "degauss": 0.002, # in Ry,
        #"tprnfor": True,
    },

    kpts=kpts,
    pseudo_dir=str(pseudo_dir),
)
atoms.calc = calc
atoms.get_potential_energy()
#log_path = run_dir / "opt.log"
#optimizer = QuasiNewton(atoms, logfile=str(log_path))
#optimizer.run(fmax=fmax)
#output_path = run_dir / structure_path.with_suffix(".relaxed_TEST.extxyz").name
#io.write(output_path, atoms, format="extxyz")
#print("relaxation complete",
#    f"structure -> {output_path}",
#    f"log -> {log_path}",
#)

-526.9492355098816